In [3]:
"""
==============================================================
DATA PREPARATION - Deteksi Stunting Posyandu
==============================================================
Dataset  : 1351 baris (balita + anak SD)
Target   : Status Stunting + Status Gizi
Model    : Random Forest Classifier
==============================================================
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ──────────────────────────────────────────────
# FASE 1: LOAD DATA
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 1: LOAD DATA")
print("=" * 55)

df = pd.read_csv('../data/raw/dataset.csv')
print(f"✅ Data berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"   Kolom: {df.columns.tolist()}\n")


# ──────────────────────────────────────────────
# FASE 2: DROP KOLOM TIDAK DIPERLUKAN
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 2: DROP KOLOM TIDAK DIPERLUKAN")
print("=" * 55)

drop_cols = ['Nama', 'Tgl Lahir', 'Nama Ortu', 'Desa/Kel', 'RT', 'RW', 'LiLA']
df.drop(columns=drop_cols, inplace=True)
print(f"✅ Kolom di-drop: {drop_cols}")
print(f"   Sisa kolom: {df.columns.tolist()}\n")


# ──────────────────────────────────────────────
# FASE 3: BERSIHKAN & KONVERSI KOLOM USIA
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 3: KONVERSI USIA KE BULAN")
print("=" * 55)

def konversi_usia_ke_bulan(usia_str):
    """
    Konversi string usia ke integer bulan.
    Contoh: '5 tahun' -> 60, '15 bulan' -> 15, '1 tahun' -> 12
    """
    if pd.isna(usia_str):
        return np.nan
    usia_str = str(usia_str).strip().lower()
    if 'tahun' in usia_str:
        try:
            tahun = int(usia_str.replace('tahun', '').strip())
            return tahun * 12
        except:
            return np.nan
    elif 'bulan' in usia_str:
        try:
            bulan = int(usia_str.replace('bulan', '').strip())
            return bulan
        except:
            return np.nan
    else:
        return np.nan

df['Usia_Bulan'] = df['Usia'].apply(konversi_usia_ke_bulan)
df.drop(columns=['Usia'], inplace=True)

print("Distribusi usia (dalam bulan) sebelum filter:")
print(df['Usia_Bulan'].value_counts().sort_index().to_string())
print()


# ──────────────────────────────────────────────
# FASE 4: FILTER HANYA BALITA (0-60 BULAN)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 4: FILTER BALITA (USIA ≤ 60 BULAN)")
print("=" * 55)

sebelum = len(df)
df = df[df['Usia_Bulan'] <= 60].copy()
sesudah = len(df)
print(f"✅ Data sebelum filter : {sebelum} baris")
print(f"✅ Data setelah filter : {sesudah} baris")
print(f"   Dibuang            : {sebelum - sesudah} baris (usia > 60 bulan)\n")


# ──────────────────────────────────────────────
# FASE 5: BERSIHKAN KOLOM JK
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 5: BERSIHKAN KOLOM JENIS KELAMIN")
print("=" * 55)

print(f"   Nilai unik sebelum : {df['JK'].unique()}")
df['JK'] = df['JK'].str.strip()
print(f"   Nilai unik sesudah : {df['JK'].unique()}\n")


# ──────────────────────────────────────────────
# FASE 6: BUAT LABEL STATUS STUNTING (dari ZS TB/U)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 6: BUAT LABEL STATUS STUNTING")
print("=" * 55)

def buat_status_stunting(zs_tbu):
    """
    Klasifikasi stunting berdasarkan standar WHO:
    ZS TB/U < -3         : Sangat Pendek
    -3 <= ZS TB/U < -2   : Pendek
    ZS TB/U >= -2        : Normal
    """
    if pd.isna(zs_tbu):
        return np.nan
    elif zs_tbu < -3:
        return 'Sangat Pendek'
    elif zs_tbu < -2:
        return 'Pendek'
    else:
        return 'Normal'

df['Status Stunting'] = df['ZS TB/U'].apply(buat_status_stunting)
print("✅ Distribusi Status Stunting:")
print(df['Status Stunting'].value_counts().to_string())
print()


# ──────────────────────────────────────────────
# FASE 7: TANGANI MISSING VALUES
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 7: TANGANI MISSING VALUES")
print("=" * 55)

print("Missing values sebelum ditangani:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())
print()

# Drop baris yang label target-nya null (ZS TB/U kosong -> Status Stunting juga kosong)
sebelum = len(df)
df.dropna(subset=['Status Stunting', 'Status Gizi', 'ZS BB/U', 'ZS BB/TB'], inplace=True)
sesudah = len(df)
print(f"✅ Baris dibuang karena label kosong: {sebelum - sesudah}")

# Isi missing pada Usia_Bulan dengan median (jika ada)
if df['Usia_Bulan'].isnull().sum() > 0:
    median_usia = df['Usia_Bulan'].median()
    df['Usia_Bulan'].fillna(median_usia, inplace=True)
    print(f"✅ Usia_Bulan yang kosong diisi dengan median: {median_usia}")

print(f"\nMissing values setelah ditangani:")
print(df.isnull().sum().to_string())
print()


# ──────────────────────────────────────────────
# FASE 8: ENCODING VARIABEL KATEGORIKAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 8: ENCODING VARIABEL KATEGORIKAL")
print("=" * 55)

# Encode JK: L=1, P=0
df['JK_encoded'] = df['JK'].map({'L': 1, 'P': 0})
print("✅ JK encoded: L=1, P=0")

# Encode Status Gizi
le_gizi = LabelEncoder()
df['Status_Gizi_encoded'] = le_gizi.fit_transform(df['Status Gizi'])
print(f"✅ Status Gizi encoded: {dict(zip(le_gizi.classes_, le_gizi.transform(le_gizi.classes_)))}")

# Encode Status Stunting
le_stunting = LabelEncoder()
df['Status_Stunting_encoded'] = le_stunting.fit_transform(df['Status Stunting'])
print(f"✅ Status Stunting encoded: {dict(zip(le_stunting.classes_, le_stunting.transform(le_stunting.classes_)))}")
print()


# ──────────────────────────────────────────────
# FASE 9: SUSUN DATASET FINAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 9: SUSUN DATASET FINAL")
print("=" * 55)

# Fitur (X)
feature_cols = ['JK_encoded', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB']

# Target stunting
X = df[feature_cols].copy()
y_stunting = df['Status_Stunting_encoded'].copy()
y_gizi     = df['Status_Gizi_encoded'].copy()

# Rename kolom agar rapi
X.columns = ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS_BBU', 'ZS_TBU', 'ZS_BBTB']

print(f"✅ Fitur (X)           : {X.columns.tolist()}")
print(f"✅ Target 1            : Status Stunting")
print(f"   Kelas              : {le_stunting.classes_.tolist()}")
print(f"✅ Target 2            : Status Gizi")
print(f"   Kelas              : {le_gizi.classes_.tolist()}")
print(f"\nShape X: {X.shape}")
print()


# ──────────────────────────────────────────────
# FASE 10: CEK CLASS IMBALANCE
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 10: CEK CLASS IMBALANCE")
print("=" * 55)

print("Distribusi Status Stunting:")
for label, count in df['Status Stunting'].value_counts().items():
    pct = count / len(df) * 100
    print(f"   {label:<20}: {count:>4} ({pct:.1f}%)")

print("\nDistribusi Status Gizi:")
for label, count in df['Status Gizi'].value_counts().items():
    pct = count / len(df) * 100
    print(f"   {label:<25}: {count:>4} ({pct:.1f}%)")
print()


# ──────────────────────────────────────────────
# FASE 11: SPLIT TRAIN & TEST
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 11: SPLIT TRAIN & TEST (80:20 Stratified)")
print("=" * 55)

# Split untuk model Stunting
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y_stunting, test_size=0.2, random_state=42, stratify=y_stunting
)

# Split untuk model Gizi
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X, y_gizi, test_size=0.2, random_state=42, stratify=y_gizi
)

print(f"✅ Model Stunting  -> Train: {X_train_s.shape[0]} | Test: {X_test_s.shape[0]}")
print(f"✅ Model Gizi      -> Train: {X_train_g.shape[0]} | Test: {X_test_g.shape[0]}\n")


# ──────────────────────────────────────────────
# FASE 12: SIMPAN DATASET BERSIH
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 12: SIMPAN DATASET")
print("=" * 55)

# Dataset lengkap bersih (dengan label asli untuk referensi)
df_clean = df[feature_cols + ['Status Stunting', 'Status Gizi']].copy()
df_clean.columns = ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS_BBU', 'ZS_TBU', 'ZS_BBTB',
                    'Status_Stunting', 'Status_Gizi']

# Simpan file
df.to_csv('../data/processed/dataset_bersih.csv', index=False)

print("✅ File tersimpan:")
print("   📄 dataset_clean.csv        → Dataset lengkap bersih")

# ──────────────────────────────────────────────
# RINGKASAN AKHIR
# ──────────────────────────────────────────────
print("=" * 55)
print("✅ DATA PREPARATION SELESAI")
print("=" * 55)
print(f"   Data awal          : 1351 baris")
print(f"   Data setelah filter: {len(df)} baris")
print(f"   Jumlah fitur       : {len(feature_cols)}")
print(f"   Target model       : 2 (Stunting & Gizi)")
print(f"   Siap untuk training Random Forest! 🚀")
print("=" * 55)

# ──────────────────────────────────────────────
# MAPPING LABEL (simpan untuk decode prediksi di web)
# ──────────────────────────────────────────────
mapping_stunting = dict(enumerate(le_stunting.classes_))
mapping_gizi     = dict(enumerate(le_gizi.classes_))

print("\nMapping label untuk decode prediksi di web:")
print(f"   Stunting : {mapping_stunting}")
print(f"   Gizi     : {mapping_gizi}")


FASE 1: LOAD DATA
✅ Data berhasil dimuat: 1351 baris, 15 kolom
   Kolom: ['Nama', 'JK', 'Usia', 'Tgl Lahir', 'Nama Ortu', 'Desa/Kel', 'RT', 'RW', 'Berat', 'Tinggi', 'LiLA', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB', 'Status Gizi']

FASE 2: DROP KOLOM TIDAK DIPERLUKAN
✅ Kolom di-drop: ['Nama', 'Tgl Lahir', 'Nama Ortu', 'Desa/Kel', 'RT', 'RW', 'LiLA']
   Sisa kolom: ['JK', 'Usia', 'Berat', 'Tinggi', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB', 'Status Gizi']

FASE 3: KONVERSI USIA KE BULAN
Distribusi usia (dalam bulan) sebelum filter:
Usia_Bulan
12      7
15      1
17      1
24     64
36    210
48    320
60    346
72    245
84    144
96     13

FASE 4: FILTER BALITA (USIA ≤ 60 BULAN)
✅ Data sebelum filter : 1351 baris
✅ Data setelah filter : 949 baris
   Dibuang            : 402 baris (usia > 60 bulan)

FASE 5: BERSIHKAN KOLOM JENIS KELAMIN
   Nilai unik sebelum : ['P\xa0' 'L\xa0' 'L' 'P']
   Nilai unik sesudah : ['P' 'L']

FASE 6: BUAT LABEL STATUS STUNTING
✅ Distribusi Status Stunting:
Status Stunting
Pen

In [ ]:
"""
==============================================================
DATA PREPARATION - Deteksi Stunting Posyandu
==============================================================
Dataset  : 1351 baris (balita + anak SD)
Target   : Status Stunting + Status Gizi
Model    : Random Forest Classifier
==============================================================
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ──────────────────────────────────────────────
# FASE 1: LOAD DATA
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 1: LOAD DATA")
print("=" * 55)

df = pd.read_csv("dataset.csv")
print(f"✅ Data berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"   Kolom: {df.columns.tolist()}\n")


# ──────────────────────────────────────────────
# FASE 2: DROP KOLOM TIDAK DIPERLUKAN
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 2: DROP KOLOM TIDAK DIPERLUKAN")
print("=" * 55)

drop_cols = ['Nama', 'Tgl Lahir', 'Nama Ortu', 'Desa/Kel', 'RT', 'RW', 'LiLA']
df.drop(columns=drop_cols, inplace=True)
print(f"✅ Kolom di-drop: {drop_cols}")
print(f"   Sisa kolom: {df.columns.tolist()}\n")


# ──────────────────────────────────────────────
# FASE 3: BERSIHKAN & KONVERSI KOLOM USIA
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 3: KONVERSI USIA KE BULAN")
print("=" * 55)

def konversi_usia_ke_bulan(usia_str):
    """
    Konversi string usia ke integer bulan.
    Contoh: '5 tahun' -> 60, '15 bulan' -> 15, '1 tahun' -> 12
    """
    if pd.isna(usia_str):
        return np.nan
    usia_str = str(usia_str).strip().lower()
    if 'tahun' in usia_str:
        try:
            tahun = int(usia_str.replace('tahun', '').strip())
            return tahun * 12
        except:
            return np.nan
    elif 'bulan' in usia_str:
        try:
            bulan = int(usia_str.replace('bulan', '').strip())
            return bulan
        except:
            return np.nan
    else:
        return np.nan

df['Usia_Bulan'] = df['Usia'].apply(konversi_usia_ke_bulan)
df.drop(columns=['Usia'], inplace=True)

print("Distribusi usia (dalam bulan) sebelum filter:")
print(df['Usia_Bulan'].value_counts().sort_index().to_string())
print()


# ──────────────────────────────────────────────
# FASE 4: FILTER HANYA BALITA (0-60 BULAN)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 4: FILTER BALITA (USIA ≤ 60 BULAN)")
print("=" * 55)

sebelum = len(df)
df = df[df['Usia_Bulan'] <= 60].copy()
sesudah = len(df)
print(f"✅ Data sebelum filter : {sebelum} baris")
print(f"✅ Data setelah filter : {sesudah} baris")
print(f"   Dibuang            : {sebelum - sesudah} baris (usia > 60 bulan)\n")


# ──────────────────────────────────────────────
# FASE 5: BERSIHKAN KOLOM JK
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 5: BERSIHKAN KOLOM JENIS KELAMIN")
print("=" * 55)

print(f"   Nilai unik sebelum : {df['JK'].unique()}")
df['JK'] = df['JK'].str.strip()
print(f"   Nilai unik sesudah : {df['JK'].unique()}\n")


# ──────────────────────────────────────────────
# FASE 6: BUAT LABEL STATUS STUNTING (dari ZS TB/U)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 6: BUAT LABEL STATUS STUNTING")
print("=" * 55)

def buat_status_stunting(zs_tbu):
    """
    Klasifikasi stunting berdasarkan standar WHO:
    ZS TB/U < -3         : Sangat Pendek
    -3 <= ZS TB/U < -2   : Pendek
    ZS TB/U >= -2        : Normal
    """
    if pd.isna(zs_tbu):
        return np.nan
    elif zs_tbu < -3:
        return 'Sangat Pendek'
    elif zs_tbu < -2:
        return 'Pendek'
    else:
        return 'Normal'

df['Status Stunting'] = df['ZS TB/U'].apply(buat_status_stunting)
print("✅ Distribusi Status Stunting:")
print(df['Status Stunting'].value_counts().to_string())
print()


# ──────────────────────────────────────────────
# FASE 7: TANGANI MISSING VALUES
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 7: TANGANI MISSING VALUES")
print("=" * 55)

print("Missing values sebelum ditangani:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())
print()

# Drop baris yang label target-nya null (ZS TB/U kosong -> Status Stunting juga kosong)
sebelum = len(df)
df.dropna(subset=['Status Stunting', 'Status Gizi', 'ZS BB/U', 'ZS BB/TB'], inplace=True)
sesudah = len(df)
print(f"✅ Baris dibuang karena label kosong: {sebelum - sesudah}")

# Isi missing pada Usia_Bulan dengan median (jika ada)
if df['Usia_Bulan'].isnull().sum() > 0:
    median_usia = df['Usia_Bulan'].median()
    df['Usia_Bulan'].fillna(median_usia, inplace=True)
    print(f"✅ Usia_Bulan yang kosong diisi dengan median: {median_usia}")

print(f"\nMissing values setelah ditangani:")
print(df.isnull().sum().to_string())
print()


# ──────────────────────────────────────────────
# FASE 8: ENCODING VARIABEL KATEGORIKAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 8: ENCODING VARIABEL KATEGORIKAL")
print("=" * 55)

# Encode JK: L=1, P=0
df['JK_encoded'] = df['JK'].map({'L': 1, 'P': 0})
print("✅ JK encoded: L=1, P=0")

# Encode Status Gizi
le_gizi = LabelEncoder()
df['Status_Gizi_encoded'] = le_gizi.fit_transform(df['Status Gizi'])
print(f"✅ Status Gizi encoded: {dict(zip(le_gizi.classes_, le_gizi.transform(le_gizi.classes_)))}")

# Encode Status Stunting
le_stunting = LabelEncoder()
df['Status_Stunting_encoded'] = le_stunting.fit_transform(df['Status Stunting'])
print(f"✅ Status Stunting encoded: {dict(zip(le_stunting.classes_, le_stunting.transform(le_stunting.classes_)))}")
print()


# ──────────────────────────────────────────────
# FASE 9: SUSUN DATASET FINAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 9: SUSUN DATASET FINAL")
print("=" * 55)

# Fitur (X)
feature_cols = ['JK_encoded', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB']

# Target stunting
X = df[feature_cols].copy()
y_stunting = df['Status_Stunting_encoded'].copy()
y_gizi     = df['Status_Gizi_encoded'].copy()

# Rename kolom agar rapi
X.columns = ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS_BBU', 'ZS_TBU', 'ZS_BBTB']

print(f"✅ Fitur (X)           : {X.columns.tolist()}")
print(f"✅ Target 1            : Status Stunting")
print(f"   Kelas              : {le_stunting.classes_.tolist()}")
print(f"✅ Target 2            : Status Gizi")
print(f"   Kelas              : {le_gizi.classes_.tolist()}")
print(f"\nShape X: {X.shape}")
print()


# ──────────────────────────────────────────────
# FASE 10: CEK CLASS IMBALANCE
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 10: CEK CLASS IMBALANCE")
print("=" * 55)

print("Distribusi Status Stunting:")
for label, count in df['Status Stunting'].value_counts().items():
    pct = count / len(df) * 100
    print(f"   {label:<20}: {count:>4} ({pct:.1f}%)")

print("\nDistribusi Status Gizi:")
for label, count in df['Status Gizi'].value_counts().items():
    pct = count / len(df) * 100
    print(f"   {label:<25}: {count:>4} ({pct:.1f}%)")
print()


# ──────────────────────────────────────────────
# FASE 11: SPLIT TRAIN & TEST
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 11: SPLIT TRAIN & TEST (80:20 Stratified)")
print("=" * 55)

# Split untuk model Stunting
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y_stunting, test_size=0.2, random_state=42, stratify=y_stunting
)

# Split untuk model Gizi
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X, y_gizi, test_size=0.2, random_state=42, stratify=y_gizi
)

print(f"✅ Model Stunting  -> Train: {X_train_s.shape[0]} | Test: {X_test_s.shape[0]}")
print(f"✅ Model Gizi      -> Train: {X_train_g.shape[0]} | Test: {X_test_g.shape[0]}\n")


# ──────────────────────────────────────────────
# FASE 12: SIMPAN DATASET BERSIH
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 12: SIMPAN DATASET")
print("=" * 55)

# Dataset lengkap bersih (dengan label asli untuk referensi)
df_clean = df[feature_cols + ['Status Stunting', 'Status Gizi']].copy()
df_clean.columns = ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS_BBU', 'ZS_TBU', 'ZS_BBTB',
                    'Status_Stunting', 'Status_Gizi']

# Simpan file
df.to_csv('../data/processed/dataset_bersih.csv', index=False)

print("✅ File tersimpan:")
print("   📄 dataset_clean.csv        → Dataset lengkap bersih")

# ──────────────────────────────────────────────
# RINGKASAN AKHIR
# ──────────────────────────────────────────────
print("=" * 55)
print("✅ DATA PREPARATION SELESAI")
print("=" * 55)
print(f"   Data awal          : 1351 baris")
print(f"   Data setelah filter: {len(df)} baris")
print(f"   Jumlah fitur       : {len(feature_cols)}")
print(f"   Target model       : 2 (Stunting & Gizi)")
print(f"   Siap untuk training Random Forest! 🚀")
print("=" * 55)

# ──────────────────────────────────────────────
# MAPPING LABEL (simpan untuk decode prediksi di web)
# ──────────────────────────────────────────────
mapping_stunting = dict(enumerate(le_stunting.classes_))
mapping_gizi     = dict(enumerate(le_gizi.classes_))

print("\nMapping label untuk decode prediksi di web:")
print(f"   Stunting : {mapping_stunting}")
print(f"   Gizi     : {mapping_gizi}")


=== ANALISIS PENYEBAB DATA TERBUANG ===
1. Total Data Mentah         : 1351
2. Terbuang (Duplikat)       : -0
3. Terbuang (Bukan Balita)   : -402
4. Terbuang (Data Kosong/NaN): -0
   DETAIL DATA KOSONG DI KOLOM PENTING:
JK             0
Berat          0
Tinggi         0
Status Gizi    0
dtype: int64
----------------------------------------
TOTAL DATA BERSIH (FINAL)    : 949


In [1]:
# ===============================
# PREDIKSI STUNTING RANDOM FOREST
# ===============================

# 1️⃣ Import library
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

# -------------------------------
# 2️⃣ Load dataset
df = pd.read_csv('../data/processed/dataset_bersih.csv')

# -------------------------------
# 3️⃣ Data prep
# Pastikan tipe data benar tanpa perlu manipulasi string lagi
df['JK'] = df['JK'].astype(int)
df['Usia'] = df['Usia'].astype(float)
df['Berat'] = df['Berat'].astype(float)
df['Tinggi'] = df['Tinggi'].astype(float)
# df['LiLA'] = df['LiLA'].astype(float)

# -------------------------------
# 4️⃣ Load data WHO & Hitung Z-Score
boys = pd.read_csv('../data/references/tab_lhfa_boys_p_0_5.csv')
girls = pd.read_csv('../data/references/tab_lhfa_girls_p_0_5.csv')

def get_who_row(usia, jk):
    table = boys if jk == 1 else girls
    # Pastikan usia dibulatkan agar cocok dengan kolom 'Month' di tabel WHO
    usia_int = int(round(usia))
    row = table.iloc[(table['Month'] - usia_int).abs().argsort()[:1]].iloc[0]
    return row

def hitung_zscore(tinggi, usia, jk):
    row = get_who_row(usia, jk)
    L, M, S = row['L'], row['M'], row['S']
    z = ((tinggi / M) ** L - 1) / (L * S)
    return z

df['Zscore_TB_U'] = df.apply(
    lambda x: hitung_zscore(x['Tinggi'], x['Usia'], x['JK']),
    axis=1
)

# -------------------------------
# 5️⃣ Buat label stunting
def klasifikasi_stunting(z):
    if z < -3: return 0   # Sangat Pendek (Severely Stunted)
    elif z < -2: return 1  # Pendek (Stunted)
    else: return 2        # Normal

df['Status_Stunting'] = df['Zscore_TB_U'].apply(klasifikasi_stunting)

# 6️⃣ Siapkan fitur & label
fitur = ['JK', 'Usia', 'Berat', 'Tinggi', 'Zscore_TB_U']
X = df[fitur]
y = df['Status_Stunting']

# -------------------------------
# 7️⃣ Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# -------------------------------
# 8. PENYEIMBANGAN DATA (Hybrid: Under + Over)
under = RandomUnderSampler(sampling_strategy={0: 300}, random_state=42)
smote = SMOTE(sampling_strategy={1: 250, 2: 250}, random_state=42)

X_res, y_res = under.fit_resample(X_train, y_train)
X_res, y_res = smote.fit_resample(X_res, y_res)

print("Jumlah data setelah diseimbangkan:")
print(pd.Series(y_res).value_counts())

# -------------------------------
# 9️⃣ Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=6, 
    min_samples_leaf=5,
    random_state=42
)
rf.fit(X_res, y_res)

# -------------------------------
# 🔟 Prediksi & evaluasi
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# -------------------------------
# 1️⃣1️⃣ Optional: Cross-validation
scores = cross_val_score(rf, X, y, cv=5)
print(f"Mean Accuracy: {scores.mean()}")


Jumlah data setelah diseimbangkan:
Status_Stunting
0    300
1    250
2    250
Name: count, dtype: int64
Accuracy: 0.9866666666666667

Confusion Matrix:
 [[132   0   0]
 [  1   3   0]
 [  0   1  13]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00       132
           1       0.75      0.75      0.75         4
           2       1.00      0.93      0.96        14

    accuracy                           0.99       150
   macro avg       0.91      0.89      0.90       150
weighted avg       0.99      0.99      0.99       150

Mean Accuracy: 0.9933154362416108


In [10]:
import joblib
# Simpan ke folder models
joblib.dump(rf, '../models/model_rf_stunting.pkl')
print("✅ Model baru dengan data seimbang sudah disimpan!")

✅ Model baru dengan data seimbang sudah disimpan!


In [12]:
# Tes Gedol dengan LiLA yang lebih kecil (sesuai rata-rata bayi)
# JK:1, Usia:25, Berat:12.4, Tinggi:88.0, LiLA:14.0
test_gedol_v2 = pd.DataFrame([[1, 25.0, 12.4, 88.0, 0]], columns=fitur)
print("Hasil Prediksi Gedol (LiLA 14.0):", rf.predict(test_gedol_v2))

Hasil Prediksi Gedol (LiLA 14.0): [2]


In [8]:
print(rf.feature_names_in_)

['JK' 'Usia' 'Berat' 'Tinggi']
